<a href="https://colab.research.google.com/github/emramushoana/SFYouTubeCodeEmra/blob/main/Cisco_onboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import tkinter as tk
from tkinter import ttk, messagebox, scrolledtext, filedialog
import serial
import serial.tools.list_ports
import threading
import time
import os

# -------- DARK THEME COLORS --------
BG = "#1e1e2f"
FG = "#ffffff"
ACCENT = "#00aaff"
ENTRY_BG = "#2a2a3d"

# -------- GLOBAL STORAGE --------
loaded_configs = []  # [(name, [lines])]
processed_configs = []  # [(name, [lines])]

# -------- SERIAL SEND WITH LOGGING --------
def send_config(port, commands, log_box, name="Single"):
    try:
        ser = serial.Serial(port, 9600, timeout=1)
        time.sleep(2)

        log_box.insert(tk.END, f"\n=== Deploying: {name} on {port} ===\n")

        for cmd in commands:
            ser.write((cmd.strip() + '\n').encode())
            time.sleep(0.7)
            output = ser.read(ser.in_waiting).decode(errors='ignore')

            log_box.insert(tk.END, f"> {cmd}\n{output}\n")
            log_box.see(tk.END)

        ser.close()
        log_box.insert(tk.END, f"=== Completed: {name} ===\n")
    except Exception as e:
        log_box.insert(tk.END, f"ERROR on {name}: {str(e)}\n")

# -------- SINGLE CONFIG (ORIGINAL FUNCTION) --------
def build_config():
    username = entry_username.get()
    password = entry_password.get()
    hostname = entry_hostname.get()
    ip = entry_ip.get()
    subnet = entry_subnet.get()
    gateway = entry_gateway.get()
    vlan = entry_vlan.get()
    port = combo_port.get()

    commands = [
        "",
        "enable",
        "configure terminal",
        f"hostname {hostname}",
        f"username {username} privilege 15 secret {password}",
        "aaa new-model",
        "aaa authentication login conlogin local",
        "aaa authentication login vtylogin local",
        f"interface vlan {vlan}",
        f"ip address {ip} {subnet}",
        "no shutdown",
        f"ip default-gateway {gateway}",
        f"ip http client source-interface Vlan{vlan}",
        f"ip ssh source-interface Vlan{vlan}",
        f"ip tacacs source-interface Vlan{vlan}",
        f"ip radius source-interface Vlan{vlan}",
        f"logging source-interface Vlan{vlan}",
        "line con 0",
        "exec-timeout 20 0",
        "login authentication conlogin",
        "line vty 0 4",
        "access-class 99 in vrf-also",
        "exec-timeout 20 0",
        "logging synchronous",
        "login authentication vtylogin",
        "transport input ssh",
        "transport output none",
        "line vty 5 15",
        "access-class 99 in vrf-also",
        "exec-timeout 20 0",
        "logging synchronous",
        "login authentication vtylogin",
        "transport input ssh",
        "transport output none",
        "ntp access-group peer NTP",
        "ntp access-group serve-only NTP",
        "ntp access-group serve-only 1310",
        "ntp server 10.127.127.1",
        "ntp server 10.219.255.123",
        "ntp server 10.217.255.123",
        "ntp server 10.0.0.123 prefer",
        "end",
        "write memory"
    ]

    threading.Thread(target=send_config, args=(port, commands, log_box, hostname)).start()

# -------- LOAD MULTIPLE CONFIGS --------
def load_configs():
    global loaded_configs
    files = filedialog.askopenfilenames(filetypes=[("Text Files", "*.txt")])

    if files:
        loaded_configs = []
        for file_path in files:
            with open(file_path, 'r') as f:
                loaded_configs.append((os.path.basename(file_path), f.readlines()))

        messagebox.showinfo("Loaded", f"Loaded {len(loaded_configs)} configs")

# -------- PROCESS CONFIGS --------
def process_configs():
    global processed_configs

    if not loaded_configs:
        messagebox.showwarning("Warning", "No configs loaded")
        return

    processed_configs = []
    preview_box.delete(1.0, tk.END)

    for name, lines in loaded_configs:
        cleaned = []

        for line in lines:
            if any(x in line for x in ["secret", "key 7", "snmp-server community"]):
                continue

            if "xxxxx" in line or "xxxx" in line:
                line = line.replace("xxxxx", "REPLACE_ME").replace("xxxx", "REPLACE_ME")

            if "login xxxxxx" in line:
                line = " login authentication vtylogin\n"

            cleaned.append(line)

        processed_configs.append((name, cleaned))

        preview_box.insert(tk.END, f"\n===== {name} =====\n")
        preview_box.insert(tk.END, "".join(cleaned))

# -------- BULK DEPLOY --------
def bulk_deploy():
    port = combo_port.get()

    if not processed_configs:
        messagebox.showwarning("Warning", "Process configs first")
        return

    def run():
        for name, config in processed_configs:
            send_config(port, config, log_box, name)
            input("Connect next switch and press Enter...")

    threading.Thread(target=run).start()

# -------- GET COM PORTS --------
def get_ports():
    return [p.device for p in serial.tools.list_ports.comports()]

# -------- GUI --------
root = tk.Tk()
root.title("Glencore Cisco Onboarding Tool")
root.geometry("950x700")
root.configure(bg=BG)

style = ttk.Style()
style.theme_use('default')

notebook = ttk.Notebook(root)
notebook.pack(fill='both', expand=True)

# -------- TAB 1: SINGLE CONFIG --------
tab1 = tk.Frame(notebook, bg=BG)
notebook.add(tab1, text="Single Config")

frame = tk.Frame(tab1, bg=BG, padx=20, pady=20)
frame.pack(fill='both', expand=True)

labels = ["Username", "Password", "Hostname", "IP Address", "Subnet Mask", "Default Gateway", "Management VLAN"]
entries = []

for i, label in enumerate(labels):
    tk.Label(frame, text=label, bg=BG, fg=FG).grid(row=i, column=0, sticky='w', pady=8)
    entry = tk.Entry(frame, bg=ENTRY_BG, fg=FG, insertbackground=FG, show="*" if label == "Password" else None)
    entry.grid(row=i, column=1, pady=8, sticky='ew')
    entries.append(entry)

entry_username, entry_password, entry_hostname, entry_ip, entry_subnet, entry_gateway, entry_vlan = entries

# COM Port
tk.Label(frame, text="COM Port", bg=BG, fg=FG).grid(row=7, column=0)
combo_port = ttk.Combobox(frame, values=get_ports())
combo_port.grid(row=7, column=1)

# Buttons
tk.Button(frame, text="Deploy", bg=ACCENT, fg=FG, command=build_config).grid(row=8, column=0, columnspan=2, pady=20)

# -------- TAB 2: BULK --------
tab2 = tk.Frame(notebook, bg=BG)
notebook.add(tab2, text="Bulk Deployment")

bulk_frame = tk.Frame(tab2, bg=BG, padx=20, pady=20)
bulk_frame.pack(fill='both', expand=True)

tk.Button(bulk_frame, text="Load Configs", bg=ACCENT, fg=FG, command=load_configs).pack(pady=10)
tk.Button(bulk_frame, text="Process", bg=ACCENT, fg=FG, command=process_configs).pack(pady=10)
tk.Button(bulk_frame, text="Bulk Deploy", bg=ACCENT, fg=FG, command=bulk_deploy).pack(pady=10)

# -------- TAB 3: PREVIEW --------
tab3 = tk.Frame(notebook, bg=BG)
notebook.add(tab3, text="Preview")

preview_box = scrolledtext.ScrolledText(tab3, bg="#121212", fg="#00ffcc")
preview_box.pack(fill='both', expand=True)

# -------- TAB 4: LOGS --------
tab4 = tk.Frame(notebook, bg=BG)
notebook.add(tab4, text="Logs")

log_box = scrolledtext.ScrolledText(tab4, bg="#121212", fg="#00ffcc")
log_box.pack(fill='both', expand=True)

root.mainloop()


TclError: no display name and no $DISPLAY environment variable